# Semantic Test vs Lakehouse — DAX Validation Against SQL Ground Truth

**Author:** vestergaardj  
**Date:** April 2026

## Abstract

This notebook validates DAX measures and queries in a Microsoft Fabric semantic model by comparing their results against SQL queries executed on a Fabric Lakehouse. Instead of manually specifying expected values, the Lakehouse serves as the **single source of truth** — if the DAX result matches the SQL result, the measure is correct.

**Problem:** After model changes (new relationships, modified measures, schema drift), DAX measures can silently return wrong results. Manually maintaining expected values for every test case doesn't scale, and there's no easy way to verify that the semantic model accurately reflects the underlying data.

**Solution:** Define test cases as pairs of DAX and SQL queries. The harness executes both, compares the resulting DataFrames, and reports pass/fail with detailed diffs. The Lakehouse data is the ground truth — if DAX diverges from SQL, something is wrong in the model.

**Key Features:**
- DataFrame-level comparison (not just scalar values)
- Automatic column name normalization (DAX `'Table'[Column]` → clean names)
- Floating-point tolerance for numeric comparisons
- Detailed diff reporting for failed tests
- Support for semantic model and lakehouse in the same or different workspaces
- Reusable with any semantic model / lakehouse pair


## How to Use This Notebook

1. **Install dependencies** — run the setup cell to install `semantic-link-labs`
2. **Configure** — set your semantic model name, workspace, and (optionally) lakehouse workspace
3. **Define test cases** — provide DAX/SQL query pairs as a list of dicts
4. **Run** — execute the test harness; it runs all test cases and compares results
5. **Review** — pass/fail summary with detailed diffs for failures

In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install semantic-link-labs


In [ ]:
import pandas as pd
import numpy as np
import datetime
import re
import sempy_labs
from IPython.display import display, Markdown, HTML

## Configuration

Set your semantic model and workspace names below. If the lakehouse resides in a different workspace, set `LAKEHOUSE_WORKSPACE` accordingly — otherwise leave it as `None` to use the same workspace.

In [ ]:
# ── Semantic Model ──────────────────────────────────────────────
DATASET_NAME = "Zendesk Report"    # Name of the semantic model
WORKSPACE_NAME = "CatMan System Data"       # Workspace containing the model

# ── Lakehouse ───────────────────────────────────────────────────
# Set to None if the lakehouse is in the same workspace as the semantic model.
# Otherwise, specify the workspace name where the lakehouse resides.
LAKEHOUSE_WORKSPACE = None

# ── Comparison Settings ─────────────────────────────────────────
DEFAULT_TOLERANCE = 0.0001   # Relative tolerance for floating-point comparison

display(Markdown(f"**Semantic Model:** `{DATASET_NAME}` in workspace `{WORKSPACE_NAME}`"))
display(Markdown(f"**Lakehouse Workspace:** `{LAKEHOUSE_WORKSPACE or WORKSPACE_NAME}` (same workspace)" if not LAKEHOUSE_WORKSPACE else f"**Lakehouse Workspace:** `{LAKEHOUSE_WORKSPACE}`"))

## Test Case Definitions

Each test case is a dict with:

| Key | Required | Description |
|-----|----------|-------------|
| `description` | Yes | Short label for the test |
| `dax_query` | Yes | A DAX `EVALUATE` query returning a table |
| `sql_query` | Yes | A Spark SQL query producing the expected result |
| `sort_columns` | No | Columns to sort by before comparison (uses all columns if omitted) |
| `tolerance` | No | Override the default floating-point tolerance for this test |
| `column_mapping` | No | Dict mapping DAX column names → SQL column names if they differ after normalization |

### Writing Good Test Pairs

- The DAX and SQL queries should return the **same shape** (same number of columns and rows)
- Column names are normalized automatically: `'Sales'[Amount]` becomes `Amount`
- Use `column_mapping` only if the normalized names still differ
- Use `sort_columns` when row order matters or for deterministic comparison

In [ ]:
# ── Define your test cases here ─────────────────────────────────
#
# Example test cases — replace with your own DAX/SQL pairs.
# These examples assume a retail-style model; adapt to your schema.

test_cases = [
    {
        "description": "Row count tickets",
        "dax_query": """
            EVALUATE
            ROW("RowCount", COUNTROWS('fact_ticket_metrics'))
        """,
        "sql_query": """
            SELECT COUNT(*) AS RowCount
            FROM Gold.fact_ticket_metrics
        """,
    },
    {
        "description": "Total # Incidents",
        "dax_query": """
            EVALUATE
            ROW("fact_ticket_metrics", [Incidents])
        """,
        "sql_query": """
            SELECT COUNT(id) AS Incidents
            FROM Gold.fact_ticket_metrics
        """,
        "tolerance": 0.01,
    },
]

print(f"Defined {len(test_cases)} test case(s).")

## Comparison Engine

Core functions that normalize column names, align DataFrames, and compare them with tolerance.

In [ ]:
def normalize_dax_column_name(col_name: str) -> str:
    """Strip DAX table qualifiers: 'Table'[Column] → Column, [Column] → Column."""
    match = re.match(r"^(?:'[^']*')?\[(.+)\]$", col_name.strip())
    return match.group(1) if match else col_name.strip()


def normalize_columns(df: pd.DataFrame, column_mapping: dict | None = None) -> pd.DataFrame:
    """Normalize DAX-style column names and apply optional mapping."""
    df = df.rename(columns={c: normalize_dax_column_name(c) for c in df.columns})
    if column_mapping:
        df = df.rename(columns=column_mapping)
    return df


def align_and_sort(dax_df: pd.DataFrame, sql_df: pd.DataFrame, sort_columns: list | None = None):
    """Align column order and sort both DataFrames consistently."""
    # Use only the intersection of columns, in alphabetical order
    common_cols = sorted(set(dax_df.columns) & set(sql_df.columns))
    extra_dax = set(dax_df.columns) - set(common_cols)
    extra_sql = set(sql_df.columns) - set(common_cols)

    dax_df = dax_df[common_cols].copy()
    sql_df = sql_df[common_cols].copy()

    # Sort
    sort_by = sort_columns if sort_columns else common_cols
    sort_by = [c for c in sort_by if c in common_cols]  # safety filter
    if sort_by:
        dax_df = dax_df.sort_values(sort_by).reset_index(drop=True)
        sql_df = sql_df.sort_values(sort_by).reset_index(drop=True)

    return dax_df, sql_df, extra_dax, extra_sql


def compare_dataframes(dax_df: pd.DataFrame, sql_df: pd.DataFrame, tolerance: float = 0.0001):
    """Compare two aligned DataFrames. Returns (passed, diff_details)."""
    if dax_df.shape != sql_df.shape:
        return False, f"Shape mismatch: DAX {dax_df.shape} vs SQL {sql_df.shape}"

    mismatches = []
    for col in dax_df.columns:
        dax_col = dax_df[col]
        sql_col = sql_df[col]

        # Try numeric comparison with tolerance
        if pd.api.types.is_numeric_dtype(dax_col) and pd.api.types.is_numeric_dtype(sql_col):
            close = np.isclose(
                dax_col.astype(float).fillna(np.nan),
                sql_col.astype(float).fillna(np.nan),
                rtol=tolerance,
                equal_nan=True,
            )
            if not close.all():
                bad_rows = np.where(~close)[0].tolist()
                for r in bad_rows[:5]:  # show first 5 mismatches per column
                    mismatches.append({
                        "column": col, "row": r,
                        "dax_value": dax_col.iloc[r],
                        "sql_value": sql_col.iloc[r],
                    })
        else:
            # String / categorical comparison
            neq = dax_col.astype(str).fillna("") != sql_col.astype(str).fillna("")
            if neq.any():
                bad_rows = np.where(neq)[0].tolist()
                for r in bad_rows[:5]:
                    mismatches.append({
                        "column": col, "row": r,
                        "dax_value": dax_col.iloc[r],
                        "sql_value": sql_col.iloc[r],
                    })

    if mismatches:
        return False, pd.DataFrame(mismatches)
    return True, None


print("Comparison engine loaded.")

## Test Harness Execution

Iterates through all test cases, executes DAX and SQL queries, normalizes results, and compares them.

In [ ]:
results = []

for i, tc in enumerate(test_cases, start=1):
    desc = tc["description"]
    dax_query = tc["dax_query"].strip()
    sql_query = tc["sql_query"].strip()
    sort_cols = tc.get("sort_columns")
    tol = tc.get("tolerance", DEFAULT_TOLERANCE)
    col_map = tc.get("column_mapping")

    print(f"\n{'='*60}")
    print(f"Test {i}/{len(test_cases)}: {desc}")
    print(f"{'='*60}")

    result = {
        "test": i,
        "description": desc,
        "passed": False,
        "error": None,
        "details": None,
        "dax_shape": None,
        "sql_shape": None,
    }

    try:
        # ── Execute DAX query ───────────────────────────────────
        print(f"  Running DAX query...")
        dax_df = sempy_labs.evaluate_dax_impersonation(
            dataset=DATASET_NAME,
            dax_query=dax_query,
            workspace=WORKSPACE_NAME,
        )
        dax_df = normalize_columns(dax_df, column_mapping=col_map)
        result["dax_shape"] = dax_df.shape
        print(f"  DAX result: {dax_df.shape[0]} rows x {dax_df.shape[1]} cols")

        # ── Execute SQL query ───────────────────────────────────
        print(f"  Running SQL query...")
        sql_df = spark.sql(sql_query).toPandas()
        result["sql_shape"] = sql_df.shape
        print(f"  SQL result: {sql_df.shape[0]} rows x {sql_df.shape[1]} cols")

        # ── Align and compare ───────────────────────────────────
        dax_aligned, sql_aligned, extra_dax, extra_sql = align_and_sort(
            dax_df, sql_df, sort_columns=sort_cols
        )

        if extra_dax:
            print(f"  ⚠ Extra columns in DAX (not in SQL): {extra_dax}")
        if extra_sql:
            print(f"  ⚠ Extra columns in SQL (not in DAX): {extra_sql}")

        passed, diff_details = compare_dataframes(dax_aligned, sql_aligned, tolerance=tol)
        result["passed"] = passed
        result["details"] = diff_details

        if passed:
            print(f"  ✅ PASSED")
        else:
            print(f"  ❌ FAILED")
            if isinstance(diff_details, pd.DataFrame):
                display(diff_details)
            else:
                print(f"     {diff_details}")

    except Exception as e:
        result["error"] = str(e)
        print(f"  ❌ ERROR: {e}")

    results.append(result)

print(f"\n{'='*60}")
print(f"All tests executed.")

## Results Summary

In [ ]:
summary_df = pd.DataFrame([
    {
        "#": r["test"],
        "Description": r["description"],
        "Result": "✅ PASS" if r["passed"] else ("⚠ ERROR" if r["error"] else "❌ FAIL"),
        "DAX Shape": str(r["dax_shape"]) if r["dax_shape"] else "-",
        "SQL Shape": str(r["sql_shape"]) if r["sql_shape"] else "-",
        "Error": r["error"] or "",
    }
    for r in results
])

num_passed = sum(1 for r in results if r["passed"])
num_failed = sum(1 for r in results if not r["passed"] and not r["error"])
num_errors = sum(1 for r in results if r["error"])

display(Markdown(f"### {num_passed} passed, {num_failed} failed, {num_errors} errors out of {len(results)} tests"))
display(summary_df)

if num_failed > 0:
    display(Markdown("---\n### ❌ Failed Test Details"))
    for r in results:
        if not r["passed"] and not r["error"] and isinstance(r["details"], pd.DataFrame):
            display(Markdown(f"**Test {r['test']}: {r['description']}**"))
            display(r["details"])

if num_failed == 0 and num_errors == 0:
    display(Markdown("### ✅ All tests passed — DAX results match Lakehouse ground truth."))

## References

- [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs) — Python library for programmatic interaction with Fabric semantic models
- [Microsoft Fabric Documentation](https://learn.microsoft.com/en-us/fabric/) — Lakehouse, Spark SQL, and semantic model documentation
- [DAX Reference](https://learn.microsoft.com/en-us/dax/) — Data Analysis Expressions language reference